# Vision pass — Stages 2 -> 2.5 -> 3 -> 4 on GPU (Colab)

Player **tracking**, **role classification**, **pose**, and **ball detection** in
one GPU trip. **Reset-proof:** each stage's outputs are backed up to Drive as it
finishes, so if the runtime disconnects you just **Run All** again and it resumes
where it left off.

The code is pulled fresh from GitHub each run — **there's no bundle to upload and
nothing to edit.** On Drive (`My Drive` root) you only need, per clip:
- `ball_model_v4.pt`  — the trained ball model (upload once)
- `<clip>_vision_input.zip`  — the clip bundle, downloaded from the app's run screen

**Runtime -> Change runtime type -> GPU**, then **Runtime -> Run all**. The clip is
auto-detected from the one `*_vision_input.zip` on Drive. Outputs land in
`content/<clip>/` and `My Drive/<clip>_outputs/`; upload them on the app run screen
to finish. Built by `tools/build_vision_nb.py`.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (SLOW — set Runtime->GPU!)')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Pull the code fresh from GitHub (clone, or fast-forward if already cloned).
import os, subprocess
REPO_URL = 'https://github.com/Hochh16/Pickleball-Analyzer-v2.git'
REPO = '/content/Pickleball-Analyzer-v2'
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO], check=True)
print('repo ready at', REPO)


In [ ]:
# Deps: Colab has torch/opencv/numpy/pandas; the stages also need these.
!pip -q install ultralytics pyarrow 2>/dev/null | tail -1
print('deps ready')


In [ ]:
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
# A re-run in the SAME session must pick up freshly pulled code: Python caches
# imported modules, so drop any previously loaded repo modules first.
for name in [n for n in list(sys.modules) if n == 'tools' or n.startswith('tools.')]:
    del sys.modules[name]
from tools.colab_vision import run_all, STAGES
print('loaded — stages:', [s['name'] for s in STAGES])


## Run

`CLIP = None` auto-detects the single clip on Drive. Only set it to a name if you
have more than one `*_vision_input.zip` uploaded.

In [ ]:
CLIP = None   # None = auto-detect the one uploaded clip; or set e.g. 'pb_5_minute_outdoor'
run_all(REPO, clip=CLIP)


## Done

Download the 7 outputs from **`content/<clip>/`** (Files panel, left) **or** from
**`My Drive/<clip>_outputs/`** (survives a runtime reset), then upload them on the
app's run screen to resume Stages 5-11 and build the report.